# Fink/LSST — Stacked histogram of alert counts vs Declination

This notebook queries the **Fink broker REST API** (LSST endpoint) live,  
retrieves the latest alerts for several classification tags, and builds a  
**stacked histogram** of the number of *diaSources* as a function of Declination (Dec).

The sky distribution directly reflects the **Rubin LSST pointing strategy**  
in effect on the query date. Declination coverage is strongly shaped by the  
observatory latitude (Cerro Pachón, –30.24°) and the Wide-Fast-Deep footprint.

---

**Column naming reminder (LSST DPDD schema)**

| Prefix | Meaning |
|--------|----------|
| `r:`   | diaSource table field (NOT the spectral band `r`) |
| `f:`   | Fink-computed field (classifiers, cross-matches) |

The spectral band is the *value* of column `r:band` ∈ `{u, g, r, i, z, y}`.

---

**Author:** dagoret  
**Date:** 2026-03-08  
**API:** https://api.lsst.fink-portal.org/api/v1

```
| **Name of Field**    | **RA(Degrees)** | **Dec (Degress)**| **Type**          |
| -------------------- | --------------- | ---------------- | ----------------- |
| **Carina**           | 161.5           | -59.7            | Galaxie/Nébuleuse |
| **ELAIS-S1**         | 9.45            | -44.0            | DDF               |
| **COSMOS**           | 150.1           | +2.2             | DDF               |
| **Trifid-Lagoon**    | 270.5           | -23.0            | Nébuleuse         |
| **M49**              | 187.4           | +8.0             | Galaxie           |
| **Rubin_SV_280_-48** | 280.0           | -48.0            | Test SV           |
| **Rubin_SV_320_-15** | 320.0           | -15.0            | Test SV           |
| **Rubin_SV_216_-17** | 216.0           | -17.0            | Test SV           |
| **Rubin_SV_212_-7**  | 212.0           | -7.0             | Test SV           |
| **Rubin_SV_225_-40** | 225.0           | -40.0            | Test SV           |
```

## 1 — Imports

In [ ]:
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime, timezone

print('Python imports OK')

## 2 — Dark-portal theme constants

In [ ]:
DARK_BG   = "#0d1117"
PANEL_BG  = "#161b22"
TEXT_COL  = "#e6edf3"
MUTED_COL = "#8b949e"
ACCENT    = "#58a6ff"
BORDER    = "#30363d"

plt.rcParams.update({
    'figure.facecolor'  : DARK_BG,
    'axes.facecolor'    : PANEL_BG,
    'axes.edgecolor'    : BORDER,
    'axes.labelcolor'   : TEXT_COL,
    'xtick.color'       : TEXT_COL,
    'ytick.color'       : TEXT_COL,
    'text.color'        : TEXT_COL,
    'grid.color'        : BORDER,
    'legend.facecolor'  : PANEL_BG,
    'legend.edgecolor'  : BORDER,
    'figure.dpi'        : 130,
})

## 3 — Configuration: tags and colours

Each entry is `(tag_name, hex_color, n_alerts_to_fetch)`.  
Increase `N_MAX` to probe a larger sky area.

In [ ]:
FINK_API = "https://api.lsst.fink-portal.org/api/v1"

# Tags to query  →  (display_label, bar_colour, n_to_fetch)
TAGS_CONFIG = [
    ("extragalactic_new_candidate",   "#58a6ff", 500),
    ("extragalactic_lt20mag_candidate","#3fb950", 500),
    ("sn_near_galaxy_candidate",      "#ff7b72", 500),
    ("in_tns",                        "#ffa657", 300),
    ("hostless_candidate",            "#a371f7", 300),
]

# Columns to retrieve — include Dec (r:dec) instead of RA
COLUMNS = "r:diaObjectId,r:dec,r:band"

BIN_WIDTH_DEG = 2   # histogram bin width in degrees

## 4 — Fetch alerts from the Fink API

In [ ]:
def fetch_tag(tag: str, n: int, columns: str = COLUMNS) -> pd.DataFrame:
    """Query /api/v1/tags and return a DataFrame."""
    url    = f"{FINK_API}/tags"
    params = {"tag": tag, "n": n, "columns": columns, "output-format": "json"}
    resp   = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data   = resp.json()
    if not data:
        print(f"  [WARNING] tag '{tag}' returned 0 rows.")
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df["tag"] = tag
    return df


frames = {}
query_time = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

for tag, colour, n_max in TAGS_CONFIG:
    print(f"Fetching  {n_max:4d}  diaSources  →  tag: {tag} ...", end=" ")
    try:
        df = fetch_tag(tag, n_max)
        frames[tag] = df
        print(f"{len(df):4d} rows")
    except Exception as exc:
        print(f"ERROR — {exc}")
        frames[tag] = pd.DataFrame()

print(f"\nQuery timestamp: {query_time}")

## 5 — Quick data summary

In [ ]:
# Print Dec range and median for each tag
for tag, colour, _ in TAGS_CONFIG:
    df = frames.get(tag, pd.DataFrame())
    if df.empty:
        print(f"{tag:<40s}  —  no data")
        continue
    dec = df["r:dec"]
    print(f"{tag:<40s}  N={len(df):4d}  "
          f"Dec=[{dec.min():7.2f}° .. {dec.max():7.2f}°]  "
          f"median={dec.median():.2f}°")

## 6 — Build the stacked histogram

In [ ]:
# ── common bin edges across all tags ──────────────────────────────────────────
all_dec = np.concatenate([
    frames[tag]["r:dec"].values
    for tag, *_ in TAGS_CONFIG
    if not frames[tag].empty
])

dec_min = np.floor(all_dec.min() / BIN_WIDTH_DEG) * BIN_WIDTH_DEG
dec_max = np.ceil( all_dec.max() / BIN_WIDTH_DEG) * BIN_WIDTH_DEG
n_bins  = int((dec_max - dec_min) / BIN_WIDTH_DEG)
bins    = np.linspace(dec_min, dec_max, n_bins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
width       = bins[1] - bins[0]

print(f"Bins: {n_bins} bins  |  Dec range: [{dec_min:.1f}°, {dec_max:.1f}°]  "
      f"|  bin width: {BIN_WIDTH_DEG}°")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

bottoms = np.zeros(n_bins)

for tag, colour, _ in TAGS_CONFIG:
    df = frames.get(tag, pd.DataFrame())
    if df.empty:
        continue
    counts, _ = np.histogram(df["r:dec"].values, bins=bins)
    ax.bar(
        bin_centers,
        counts,
        width=width * 0.92,
        bottom=bottoms,
        color=colour,
        label=tag,
        alpha=0.85,
        linewidth=0,
    )
    bottoms += counts

# ── Rubin/LSST observatory latitude (Cerro Pachón) ────────────────────────────
obs_lat = -30.24
ax.axvline(
    obs_lat, color="#f0e68c", linewidth=1.2, linestyle="--",
    label=f"Observatory latitude ({obs_lat:.2f}°)"
)

# ── Galactic equator (b = 0°) crossing hint around Dec ≈ –29° ─────────────────
# (approximate galactic-plane Dec at the Galactic centre direction)
ax.axvspan(-35, -25, alpha=0.08, color="#ffffff", label="Galactic plane region (~b=0)")

# ── cosmetic ──────────────────────────────────────────────────────────────────
ax.set_xlabel("Declination (deg)", fontsize=13)
ax.set_ylabel("Number of diaSources", fontsize=13)
ax.set_title(
    f"Fink/LSST — Alert count vs Declination\n"
    f"(stacked by classification tag, {query_time})",
    fontsize=14,
)

ax.set_xlim(dec_min - BIN_WIDTH_DEG, dec_max + BIN_WIDTH_DEG)
ax.xaxis.set_major_locator(mticker.MultipleLocator(10))
ax.xaxis.set_minor_locator(mticker.MultipleLocator(2))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax.grid(axis="y", alpha=0.3)

ax.legend(
    fontsize=9,
    loc="upper left",
    framealpha=0.7,
)

fig.tight_layout()
plt.show()

## 7 — Save the figure

In [ ]:
outpath = "fink_dec_stacked_histogram.png"
fig.savefig(outpath, dpi=150, bbox_inches="tight", facecolor=DARK_BG)
print(f"Figure saved → {outpath}")

## 8 — (Optional) Per-tag Dec distributions — side-by-side panels

In [ ]:
active_tags = [(tag, colour) for tag, colour, _ in TAGS_CONFIG
               if not frames.get(tag, pd.DataFrame()).empty]

n_tags = len(active_tags)
ncols  = min(3, n_tags)
nrows  = (n_tags + ncols - 1) // ncols

fig2, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows),
                           sharey=False, sharex=True)
axes_flat = np.array(axes).flatten()

for i, (tag, colour) in enumerate(active_tags):
    ax_ = axes_flat[i]
    df  = frames[tag]
    counts, _ = np.histogram(df["r:dec"].values, bins=bins)
    ax_.bar(bin_centers, counts, width=width * 0.92,
            color=colour, alpha=0.85, linewidth=0)
    ax_.axvline(obs_lat, color="#f0e68c", linewidth=1.0, linestyle="--")
    ax_.set_title(tag, fontsize=9)
    ax_.set_xlabel("Dec (deg)", fontsize=8)
    ax_.set_ylabel("Count", fontsize=8)
    ax_.grid(axis="y", alpha=0.3)
    ax_.xaxis.set_major_locator(mticker.MultipleLocator(10))

# Hide unused subplots
for j in range(n_tags, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig2.suptitle(
    f"Fink/LSST — Dec distribution per tag ({query_time})",
    fontsize=13, y=1.01
)
fig2.tight_layout()
plt.show()

## 9 — (Optional) Stacked histogram coloured by spectral band

Here we stack on the *band* (`r:band`) dimension instead of tag,  
to see which Rubin filters contribute most at each Declination.

In [ ]:
BAND_COLORS = {
    "u": "#2c7bb6",   # blue
    "g": "#1a9641",   # green
    "r": "#d7191c",   # red
    "i": "#fdae61",   # orange-yellow
    "z": "#7b3294",   # purple
    "y": "#c51b7d",   # magenta
}

MARKERS = {
    "u": "o",
    "g": "s",
    "r": "D",
    "i": "^",
    "z": "v",
    "y": "P",
}

# Merge all tags into one DataFrame and keep only rows with a known band
df_all = pd.concat(
    [frames[tag] for tag, *_ in TAGS_CONFIG if not frames[tag].empty],
    ignore_index=True,
)
df_all = df_all[df_all["r:band"].isin(BAND_COLORS)].copy()

present_bands = [b for b in ["u", "g", "r", "i", "z", "y"] if b in df_all["r:band"].values]
print(f"Bands present: {present_bands}  |  Total rows: {len(df_all):,}")

In [ ]:
# ── common bin edges for the full merged dataset ───────────────────────────────
dec_vals = df_all["r:dec"].values
dec_min_b = np.floor(dec_vals.min() / BIN_WIDTH_DEG) * BIN_WIDTH_DEG
dec_max_b = np.ceil( dec_vals.max() / BIN_WIDTH_DEG) * BIN_WIDTH_DEG
n_bins_b  = int((dec_max_b - dec_min_b) / BIN_WIDTH_DEG)
bins_b    = np.linspace(dec_min_b, dec_max_b, n_bins_b + 1)
centers_b = 0.5 * (bins_b[:-1] + bins_b[1:])
width_b   = bins_b[1] - bins_b[0]

fig3, ax3 = plt.subplots(figsize=(14, 6))

bottoms_b = np.zeros(n_bins_b)

for band in present_bands:
    sub    = df_all[df_all["r:band"] == band]["r:dec"].values
    counts, _ = np.histogram(sub, bins=bins_b)
    ax3.bar(
        centers_b, counts,
        width=width_b * 0.92,
        bottom=bottoms_b,
        color=BAND_COLORS[band],
        label=f"band {band}",
        alpha=0.85,
        linewidth=0,
    )
    bottoms_b += counts

ax3.axvline(obs_lat, color="#f0e68c", linewidth=1.2, linestyle="--",
            label=f"Observatory latitude ({obs_lat:.2f}°)")

ax3.set_xlabel("Declination (deg)", fontsize=13)
ax3.set_ylabel("Number of diaSources", fontsize=13)
ax3.set_title(
    f"Fink/LSST — Alert count vs Declination\n"
    f"(stacked by spectral band, {query_time})",
    fontsize=14,
)
ax3.set_xlim(dec_min_b - BIN_WIDTH_DEG, dec_max_b + BIN_WIDTH_DEG)
ax3.xaxis.set_major_locator(mticker.MultipleLocator(10))
ax3.xaxis.set_minor_locator(mticker.MultipleLocator(2))
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
ax3.grid(axis="y", alpha=0.3)
ax3.legend(fontsize=9, loc="upper left", framealpha=0.7)

fig3.tight_layout()
plt.show()

```
| **Name of Field**    | **RA(Degrees)** | **Dec (Degress)**| **Type**          |
| -------------------- | --------------- | ---------------- | ----------------- |
| **Carina**           | 161.5           | -59.7            | Galaxie/Nébuleuse |
| **ELAIS-S1**         | 9.45            | -44.0            | DDF               |
| **COSMOS**           | 150.1           | +2.2             | DDF               |
| **Trifid-Lagoon**    | 270.5           | -23.0            | Nébuleuse         |
| **M49**              | 187.4           | +8.0             | Galaxie           |
| **Rubin_SV_280_-48** | 280.0           | -48.0            | Test SV           |
| **Rubin_SV_320_-15** | 320.0           | -15.0            | Test SV           |
| **Rubin_SV_216_-17** | 216.0           | -17.0            | Test SV           |
| **Rubin_SV_212_-7**  | 212.0           | -7.0             | Test SV           |
| **Rubin_SV_225_-40** | 225.0           | -40.0            | Test SV           |
```